In [164]:
import abc
import math
import random as rd
import typing as tp

import numpy as np

In [165]:
class Arm(tp.Protocol):
    def pull(self) -> int:
        pass

    def get_prob(self) -> float:
        pass

In [166]:
class BanditStrategy(tp.Protocol):
    def get_counts(self) -> list[float]:
        pass

    def get_values(self) -> list[float]:
        pass

    def get_total_reward(self) -> int:
        pass

    def update(self, n_arm: int, reward: int) -> None:
        pass

    def choose(self) -> int:
        pass

In [167]:
class SlotMachine(Arm):
    def __init__(self, name: str, probability: float) -> None:
        self._name = name
        self._probability = probability

    def pull(self) -> int:
        return 1 if rd.random() < self._probability else 0

    def get_prob(self) -> float:
        return self._probability

    def __str__(self) -> str:
        return self._name

In [168]:
class BaseBanditStrategy(BanditStrategy):
    def __init__(self, n_arms: int) -> None:
        self._n_arms = n_arms
        self._counts = np.zeros(self._n_arms)
        self._values = np.zeros(self._n_arms)
        self._total_reward = 0

    def get_counts(self) -> list[float]:
        return self._counts.tolist()

    def get_values(self) -> list[float]:
        return self._values.tolist()

    def get_total_reward(self) -> int:
        return self._total_reward

    def update(self, n_arm: int, reward: int) -> None:
        self._update_counts(n_arm)
        self._update_values(n_arm, reward)
        self._update_cum_reward(reward)

    @abc.abstractmethod
    def choose(self) -> int:
        pass

    def _update_values(self, n_arm: int, reward: int) -> None:
        self._values[n_arm] += (reward - self._values[n_arm]) / self._counts[n_arm]

    def _update_counts(self, n_arm: int) -> None:
        self._counts[n_arm] += 1

    def _update_cum_reward(self, reward: int) -> None:
        self._total_reward += reward

    def __str__(self) -> str:
        return self.__class__.__name__

In [169]:
class RandomBanditStrategy(BaseBanditStrategy):
    def choose(self) -> int:
        return np.random.randint(self._n_arms)


In [170]:
class GreedyBanditStrategy(BaseBanditStrategy):
    def choose(self) -> int:
        if np.min(self._counts) == 0:
            return np.argmin(self._counts)
        return np.argmax(self._values)

In [171]:
class EpsilonRandomGreedyBanditStrategy(BaseBanditStrategy):
    def __init__(self, n_arms: int, epsilon: float) -> None:
        super().__init__(n_arms)
        self._epsilon = epsilon

    def choose(self) -> int:
        if np.random.random() < self._epsilon:
            return np.random.randint(self._n_arms)
        return np.argmax(self._values)

In [172]:
def test_bandit_strategy(strategy: BanditStrategy, arms: tp.Sequence[Arm], attempts: int) -> None:
    for _ in range(attempts):
        n_arm = strategy.choose()
        reward = arms[n_arm].pull()
        strategy.update(n_arm, reward)

In [173]:
def calculate_regret_perc(strategy: BanditStrategy, arms: tp.Sequence[Arm], attempts: int) -> float:
    best_prob = max(arm.get_prob() for arm in arms)
    if best_prob == 0:
        return 0
    best_total_reward = best_prob * attempts
    actual_total_reward = strategy.get_total_reward()
    regret = best_total_reward - actual_total_reward
    return regret / best_total_reward

In [174]:
def print_results(strategy: BanditStrategy, arms: tp.Sequence[Arm], regret: float) -> None:
    values = strategy.get_values()
    print(strategy)
    print("Arm | Regret %")
    for i, arm in enumerate(arms):
        print(f"{str(arm):^3} | {values[i]:.2f}")
    print(f"Regret: {regret:.2f}")
    print()

In [175]:
arms = [
    SlotMachine("A", 0.1),
    SlotMachine("B", 0.2),
    SlotMachine("C", 0.3),
]
n_arms = len(arms)
attempts = 1000
epsilon = 0.1
strategies = [
    RandomBanditStrategy(n_arms),
    GreedyBanditStrategy(n_arms),
    EpsilonRandomGreedyBanditStrategy(n_arms, epsilon),
]

for strategy in strategies:
    test_bandit_strategy(strategy, arms, attempts)
    regret = calculate_regret_perc(strategy, arms, attempts)
    print_results(strategy, arms, regret)
    calculate_regret_perc(strategy, arms, attempts)

RandomBanditStrategy
Arm | Regret %
 A  | 0.08
 B  | 0.22
 C  | 0.35
Regret: 0.28

GreedyBanditStrategy
Arm | Regret %
 A  | 0.00
 B  | 0.21
 C  | 0.00
Regret: 0.32

EpsilonRandomGreedyBanditStrategy
Arm | Regret %
 A  | 0.15
 B  | 0.19
 C  | 0.32
Regret: 0.17



## Выбор статического Epsilon
Epsilon -> 1.00 когда (мало попыток) + (варианты близки по вероятностям) + (вероятности динамическиe)
Epsilon -> 0.00 когда (много попыток) + (варианты различны по вероятностям) + (вероятности статичны)

## UCB (Upper Confidence Bound)
Даем каждому варианту **бонус за неопределенность**. Чем меньше мы его пробовали - тем больше бонус. Потом выбираем вариант с максимальной оценкой + бонус.
```
choose(n_arm) = среднее_вознаграждение + √(2 × ln(total_attempts) / n_arm_attempts)
                 ↑                          ↑
            эксплуатация              бонус за исследование
```

Где:
- `total_attempts` = общее кол-во потраченных попыток
- `n_arm_attempts` = кол-во попыток потраченных на нынешний вариант


In [180]:
class UCBBanditStrategy(BaseBanditStrategy):
    def choose(self) -> int:
        if np.min(self._counts) == 0:
            return np.argmin(self._counts)
        return max(range(n_arms), key=self._calculate_ucb)

    def _calculate_ucb(self, n_arm) -> float:
        return self._values[n_arm] + math.sqrt(2 * math.log(self._n_arms) / self._counts[n_arm])

In [185]:
arms = [
    SlotMachine("A", 0.1),
    SlotMachine("B", 0.2),
    SlotMachine("C", 0.3),
    SlotMachine("D", 0.6),
]
n_arms = len(arms)
attempts = 10000
epsilon = 0.1
strategies = [
    RandomBanditStrategy(n_arms),
    GreedyBanditStrategy(n_arms),
    EpsilonRandomGreedyBanditStrategy(n_arms, epsilon),
    UCBBanditStrategy(n_arms)
]

for strategy in strategies:
    test_bandit_strategy(strategy, arms, attempts)
    regret = calculate_regret_perc(strategy, arms, attempts)
    print_results(strategy, arms, regret)
    calculate_regret_perc(strategy, arms, attempts)

RandomBanditStrategy
Arm | Regret %
 A  | 0.10
 B  | 0.21
 C  | 0.29
 D  | 0.61
Regret: 0.49

GreedyBanditStrategy
Arm | Regret %
 A  | 0.10
 B  | 0.00
 C  | 0.00
 D  | 0.00
Regret: 0.84

EpsilonRandomGreedyBanditStrategy
Arm | Regret %
 A  | 0.10
 B  | 0.14
 C  | 0.28
 D  | 0.60
Regret: 0.06

UCBBanditStrategy
Arm | Regret %
 A  | 0.00
 B  | 0.14
 C  | 0.30
 D  | 0.60
Regret: 0.01

